# Malaysia Household Resilience Monitor

This notebook reproduces the main analysis from official Malaysian public data.

**Policy question:** Are household incomes across Malaysian states keeping pace with living costs, and where do multiple indicators suggest a need for closer policy attention?

The notebook intentionally separates household income, prices, poverty and formal wages rather than collapsing them into a black-box score.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parents[0]
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from build_indicators import build_household_metrics, build_wage_pulse


## 1. Load public source data

Run `python src/fetch_data.py` before this notebook. Raw datasets are not committed to GitHub.

In [ ]:
RAW = ROOT / "data" / "raw"

hies = pd.read_csv(RAW / "hies_state.csv")
cpi = pd.read_csv(RAW / "cpi_state.csv")
wages = pd.read_csv(RAW / "formal_wages_state.csv")

hies.head(), cpi.head(), wages.head()


## 2. Construct transparent indicators

The main real-income calculation is:

\[
g_{real} = \frac{1 + g_{nominal}}{1 + \pi} - 1
\]

where `g_nominal` is median household income growth and `π` is the change in annual-average state CPI.

In [ ]:
household = build_household_metrics(hies, cpi)
wage_pulse = build_wage_pulse(wages)

metrics = household.merge(
    wage_pulse[["state", "wages_median_latest", "wage_yoy_growth", "wage_latest_date"]],
    on="state",
    how="left",
)
metrics.sort_values("real_median_income_growth").head(10)


## 3. Inspect evidence boundaries

The triage label is not a welfare ranking. It flags observations that combine **above-median poverty** with **below-median real income growth**. A 'watch' state meets one of those conditions.

In [ ]:
metrics.groupby("policy_triage", dropna=False).size()


In [ ]:
plot_df = metrics.sort_values("real_median_income_growth")

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(plot_df["state"], plot_df["real_median_income_growth"] * 100)
ax.set_xlabel("Real median household income growth (%)")
ax.set_title("Real Median Household Income Growth Between Latest HIES Years")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(metrics["real_median_income_growth"] * 100, metrics["poverty_latest"])

for _, row in metrics.iterrows():
    ax.annotate(
        row["state"],
        (row["real_median_income_growth"] * 100, row["poverty_latest"]),
        fontsize=7,
        xytext=(3, 3),
        textcoords="offset points",
    )

ax.axvline(metrics["real_median_income_growth"].median() * 100, linestyle="--", linewidth=1)
ax.axhline(metrics["poverty_latest"].median(), linestyle="--", linewidth=1)
ax.set_xlabel("Real median household income growth (%)")
ax.set_ylabel("Latest poverty rate (%)")
ax.set_title("Policy Triage: Poverty and Real Income Growth")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 4. Policy interpretation

A useful next step is not to ask *which state is worst?* but:

- Which states combine weak real income growth with higher poverty?
- Are those patterns concentrated in particular household or labour-market groups?
- Do food, housing or transport prices explain more of the pressure than overall CPI?
- Are formal wage gains consistent with household income changes?
- Which findings remain after revised data are released?

Those questions convert the screening analysis into a research agenda rather than an unsupported ranking.